In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')


In [2]:
#load data
df = pd.read_fwf("armada.dat")
df.head()

df.columns = [
    "Battle", 
    "Year", 
    "Portuguese ships", 
    "Dutch ships", 
    "English ships", 
    "Ratio", 
    "Spanish Involvement", 
    "Portuguese outcome"
]
df

,Battle,Year,Portuguese ships,Dutch ships,English ships,Ratio,Spanish Involvement,Portuguese outcome
0,Malacca Strait,1606,14,11,0,1.273,0,0
1,Ilha das Naus,1606,6,9,0,0.667,0,-1
2,Pulo Butum,1606,7,9,0,0.778,0,1
3,Surrat,1615,6,0,4,1.500,0,0
4,Ilha das Naus,1615,3,5,0,0.600,0,-1
5,Jask,1620,4,0,4,1.000,0,0
6,Hormuz,1622,6,0,5,1.200,0,-1
7,Mogincoal Shoals,1622,4,4,2,0.667,0,-1
8,Hormuz,1625,8,4,4,1.000,0,0
9,Goa,1636,6,4,0,1.500,0,0


In [3]:
#select features and set target
X = df[['Portuguese ships', 'Dutch ships', 'English ships', 'Ratio', 'Spanish Involvement']]
y = df['Portuguese outcome']

In [4]:
#test-train split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
#scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
#Model 1 Support Vector Machine
param_grid = {
    'C': [0.1, 1, 10, 50, 100],
    'gamma': ['scale', 'auto', 0.01, 0.001, 0.0001],
    'kernel': ['rbf', 'poly', 'sigmoid']
}

svm = SVC()

#grid-search 
grid = GridSearchCV(
    svm,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)

print("Best Params:", grid.best_params_)
print("Best Cross Validation Score:", grid.best_score_)

best_svm = grid.best_estimator_

# Predictions
svm_preds = best_svm.predict(X_test_scaled)

print("\n=== Tuned SVM =====")
print("Accuracy:", accuracy_score(y_test, svm_preds))
print("Macro F1:", f1_score(y_test, svm_preds, average="macro"))
print("\nClassification Report:\n", classification_report(y_test, svm_preds))

Best Params: {'C': 100, 'gamma': 'auto', 'kernel': 'sigmoid'}
Best Cross Validation Score: 0.32222222222222224

=== Tuned SVM =====
Accuracy: 0.5
Macro F1: 0.38888888888888884

Classification Report:
               precision    recall  f1-score   support

          -1       0.50      1.00      0.67         2
           0       1.00      0.33      0.50         3
           1       0.00      0.00      0.00         1

    accuracy                           0.50         6
   macro avg       0.50      0.44      0.39         6
weighted avg       0.67      0.50      0.47         6



GridSearchCV selected an SVM with C = 100, gamma = ‘auto’, and a sigmoid kernel, which achieved a best cross-validation macro F1 of 0.32.
The choice of a sigmoid kernel suggests that the dataset does not exhibit strong nonlinear boundaries.The very high C value indicates the model is attempting to fit noise in the small dataset.

On the test set, the tuned SVM achieved 50% accuracy and a macro F1 of 0.39. The model performs reasonably on the “defeat” class but fails to correctly classify the “victory” class (recall = 0). This behavior is expected given the small sample size, three-class structure, and class imbalance of the Portuguese sea battles dataset. Overall, the SVM struggled to generalize, indicating that the available battle features (ship counts, ratios, and Spanish involvement) do not provide strong separability for the three outcome categories.


Reference: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

In [7]:
#Model 2 - KNN
param_grid = {
    'n_neighbors': [1, 2, 3, 4, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn = KNeighborsClassifier()

grid_knn = GridSearchCV(
    knn,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

# Fit on scaled data
grid_knn.fit(X_train_scaled, y_train)

print("Best Params:", grid_knn.best_params_)
print("Best CV Score:", grid_knn.best_score_)

# Best model
best_knn = grid_knn.best_estimator_

# Predictions
knn_preds = best_knn.predict(X_test_scaled)

print("\n===== Tuned KNN =====")
print("Accuracy:", accuracy_score(y_test, knn_preds))
print("Macro F1:", f1_score(y_test, knn_preds, average="macro"))
print("\nClassification Report:\n", classification_report(y_test, knn_preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, knn_preds))


Best Params: {'metric': 'euclidean', 'n_neighbors': 4, 'weights': 'uniform'}
Best CV Score: 0.32666666666666666

===== Tuned KNN =====
Accuracy: 0.6666666666666666
Macro F1: 0.48888888888888893

Classification Report:
               precision    recall  f1-score   support

          -1       0.50      1.00      0.67         2
           0       1.00      0.67      0.80         3
           1       0.00      0.00      0.00         1

    accuracy                           0.67         6
   macro avg       0.50      0.56      0.49         6
weighted avg       0.67      0.67      0.62         6


Confusion Matrix:
 [[2 0 0]
 [1 2 0]
 [1 0 0]]


The tuned KNN model selected k = 4, Euclidean distance, and uniform weighting as the optimal configuration, achieving a best cross-validated macro F1 of 0.33. This indicates modest separation among the three outcome classes, consistent with the limitations of the dataset. On the test set, the tuned KNN achieved 66.7% accuracy and a macro F1 of 0.49, outperforming the SVM. The model performed well on defeats and draws but was unable to identify the rare victory class, which had only a single example in the test split. The confusion matrix shows that most mistakes occur between draws and defeats, suggesting that fleet sizes and ratios do not provide strong discriminative signals for predicting decisive victories. Overall, KNN offers slightly better generalization than the SVM for this small, imbalanced historical dataset.

In [8]:
#Model 3 - Decision Tree
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

dt_preds = dt.predict(X_test)

print("===== Decision Tree =====")
print("Accuracy:", accuracy_score(y_test, dt_preds))
print("Macro F1:", f1_score(y_test, dt_preds, average="macro"))
print("\nClassification Report:\n", classification_report(y_test, dt_preds))


===== Decision Tree =====
Accuracy: 0.8333333333333334
Macro F1: 0.6190476190476191

Classification Report:
               precision    recall  f1-score   support

          -1       1.00      1.00      1.00         2
           0       0.75      1.00      0.86         3
           1       0.00      0.00      0.00         1

    accuracy                           0.83         6
   macro avg       0.58      0.67      0.62         6
weighted avg       0.71      0.83      0.76         6



The Decision Tree classifier achieved the highest accuracy (83%) among the models tested. It perfectly classified both defeats (-1) and all draw outcomes (0), indicating that the model was able to extract clear rule-based patterns from ship numbers and involvement features. However, similar to KNN and SVM, the tree could not correctly identify the single “victory” (class 1) example in the test set, reflecting the severe class imbalance and the limited number of training samples for that category. Despite this limitation, the Decision Tree achieved a strong weighted F1 score (0.76) and robust overall performance, demonstrating that rule-based learners can capture meaningful structure in this small historical dataset more effectively than margin-based (SVM) or distance-based (KNN) approaches.